# Часть 1. Обработка категориальных переменных

Как мы знаем, перекодировать категориальную переменную в список чисел (к примеру 1, 2, 3, ..., n) плохо, поскольку это бы задало на множестве ее значений некоторый порядок, не имеющий смысла.

В этой части мы рассмотрим два основных способа обработки категориальных значений:
- One-hot-кодирование
- Счётчики (CTR, mean-target кодирование, ...) — каждый категориальный признак заменяется на среднее значение целевой переменной по всем объектам, имеющим одинаковое значение в этом признаке.

Начнём с one-hot-кодирования. Допустим наш категориальный признак $f_j(x)$ принимает значения из множества $C=\{c_1, \dots, c_m\}$. Заменим его на $m$ бинарных признаков $b_1(x), \dots, b_m(x)$, каждый из которых является индикатором одного из возможных категориальных значений:
$$
b_i(x) = [f_j(x) = c_i]
$$

#### __Подготовка данных__

(бесценный шаг)

Загрузим данные [UCI Adult Dataset](https://archive.ics.uci.edu/ml/datasets/Adult). Этот набор данных содержит информацию о годовых доходах отдельных людей. В качестве признакового описания используется различная информация о человеке (образование, профессия, брачный статус и т.д.). Целевая переменная является бинарной: больше ли годовой доход 50K долларов или нет.

In [ ]:
#!wget https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:

columns = ['age', 'workclass', 'fnlwgt', 'education',
           'education-num', 'marital-status', 'occupation',
           'relationship', 'race', 'sex', 'capital-gain',
           'capital-loss', 'hours-per-week', 'native-country',
           'income']

df = pd.read_csv('adult_data.csv', header=None, names=columns)
df['income'] = (df['income'] != " <=50K").astype('int32')

In [3]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


Разделите выборку на обучающую и тестовую в соотношении 3:1. Зафиксируйте `random_state=777`, также используйте `stratify`.

In [ ]:
#your code here

#### __Задание 3. OrdinalEncoder__  (0.5 балла)

Закодируйте категориальные признаки с помощью `OrdinalEncoder`, а числовые признаки нормализуйте с помощью `StandardScaler`. Посчитайте качество (в этом задании будем работать c __`accuracy`__) при применении логистической регрессии. Замерьте время, потребовавшееся на обучение модели, с учетом кодирования признаков.

In [5]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
encoder.fit_transform(X_train)

#### __Задание 4. One-Hot Encoding__ (0.5 балла)



Закодируйте все категориальные признаки с помощью one-hot-кодирования. Обучите логистическую регрессию и посмотрите, как изменилось качество модели (в сравнении с тем, что было до кодирования). Измерьте время, потребовавшееся на кодирование категориальных признаков и обучение модели.

In [ ]:
#your code here

Как можно заметить, one-hot-кодирование может сильно увеличивать количество признаков. Это сказывается на объеме необходимой памяти, особенно, если некоторый признак имеет большое количество значений.


#### __Задание 5. Mean-target Encoding__ (1 балл)

> Проблемы разрастания числа признаков можно избежать в другом способе кодирования категориальных признаков — mean-target encoding (для простоты будем называть это __счётчиками__). Сравним эффективность методов в рамках нашей маркетинговой задачи.

> Основная идея в том, что важны не сами категории, а значения целевой переменной, которые имеют объекты этой категории. Каждый категориальный признак мы заменим средним значением целевой переменной по всем объектам этой же категории:
$$
g_j(x, X) = \frac{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)][y_i = +1]}{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)]}
$$

Закодируйте категориальные переменные с помощью счётчиков (ровно так, как описано выше, без каких-либо хитростей). Обучите логистическую регрессию и посмотрите на качество модели на тестовом множестве.

Для кодирования используйте [TargetEncoder из sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html).

In [ ]:
#your code here

_______

__Методы борьбы с переобучением счетчиков__


Отметим, что mean-target encoding признаки сами по себе являются классификаторами и, обучаясь на них, мы допускаем "утечку" целевой переменной в признаки. Это ведёт к __переобучению__, поэтому считать такие признаки необходимо таким образом, чтобы при вычислении для конкретного объекта его __целевая метка не использовалась__.

Это можно делать следующими способами:
1. Вычислять значение счётчика по всем объектам расположенным выше в датасете (например, если у нас выборка отсортирована по времени).
2. Вычислять по фолдам, то есть делить выборку на некоторое количество частей и подсчитывать значение признаков по всем фолдам кроме текущего (как делается в кросс-валидации).
3. Внесение некоторого шума в посчитанные признаки.

#### __Задание 7. Сглаживание счетчиков__  (1 балл)

> Теперь ответим на следующий вопрос: что будет, если некоторая категория встречается в выборке всего несколько раз? По этой причине производится сглаживание счётчиков. Например, на практике хорошие результаты показывает использование сглаживания средним по всей выборке:
$$
g_j(x, X) = \frac{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)][y_i = +1] + C \times global\_mean}{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)] + C}
$$
где $global\_mean$ — доля объектов положительного класса в выборке, $C$ — параметр, определяющий степень сглаживания (можно использовать 10 или подобрать для каждого признака свой). Идея в том, что мы "разбавляем" среднее значение по категории глобальным средним значением. И тем меньше, чем большее количество объектов этой категории встречается в выборке.

> Вместо среднего значения целевой переменной для сглаживания можно использовать любое другое значение от 0 до 1 (этот параметр иногда называют $prior$). Можно сделать несколько признаков с разными значениями параметра. На практике в задачах бинарной классификации полезными бывают даже отрицательные значения!

Добавьте сглаживание, описанное выше и повторите эксперименты. Внимательно посмотрите на параметры [TargetEncoder из sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html). Позволяет ли он сделать это быстро?


In [ ]:
#your code here

#### __Задание 8. И все-таки числовой?__  (0.5 балла)

В данных имеются признаки "возраст" и "число рабочих часов в неделю". Сейчас мы интерпретируем их как числовые, что в общем случае для линейной модели может быть неверной гипотезой. Тем не менее, у этих признаков есть довольно много уникальных значений (сколько?), поэтому применять к ним one-hot кодирование может оказаться излишним. Попробуйте закодировать эти признаки с помощью счетчиков (вместе и по отдельности). Стало ли лучше?



In [ ]:
#your code here

> __Замечание.__ Усложнение методов вычисления счётчиков не делают результаты модели гарантированно лучше. Особенно с учётом того, что логистическая регрессия не такая сложная модель, чтобы переобучаться. Поэтому вы необязательно должны были получать на каждом шаге всё лучшие и лучшие результаты (но необходимые результаты у вас должны были получиться).

Как мы могли пронаблюдать, счётчики являются конкурентной альтернативой one-hot-кодированию. Опишите, какие плюсы и минусы использования счётчиков по сравнению с one-hot-кодированием вы заметили.

__Ответ:__ # your answer here

# Часть 2. Отбор признаков

Важной частью процесса построения модели является отбор признаков. На практике многие признаки оказывают малое влияние на модель (при этом их расчёт занимает время) или даже негативно сказываются на качестве модели. Попробуем несколько подходов отбора признаков, оценим, как они влияют на качество модели и сколько времени занимают.

Обратимся к тем же данным про предсказание дохода.

In [ ]:
columns = ['age', 'workclass', 'fnlwgt', 'education',
           'education-num', 'marital-status', 'occupation',
           'relationship', 'race', 'sex', 'capital-gain',
           'capital-loss', 'hours-per-week', 'native-country',
           'income']

df = pd.read_csv('adult.data', header=None, names=columns)
df['income'] = (df['income'] != " <=50K").astype('int32')

Разделите выборку на обучающую и тестовую в соотношении 3:1. Зафиксируйте `random_state=777`, также используйте `stratify`.

In [ ]:
#your code here

Давайте закодируем все категориальные признаки с помощью One-hot Encoding, считая возраст и число часов числовыми. Сколько новых признаков мы получим?

In [ ]:
#your code here

В качестве основной модели будем использовать логистическую регрессию, а целевой метрики - `accuracy`. Обучите модель и посчитайте качество на тестовой выборке. Давайте запомним полученное значение.

In [ ]:
#your code here

#### __Задание 9. Встроенные методы (0.5 балла)__

Допустим, мы хотим оставить только 40 лучших признаков. Попробуем сделать это несколькими способами.

Начнём с отборам признаков с помощью линейной модели. Как известно, веса линейной модели означают вклад каждого признака в предсказание модели, а значит, модуль этого вклада можно интерпретировать как важность признаков. Такой метод отбора называются встроенным или embedded method, так как он заложен в особенности модели.

Оставьте 40 признаков с наибольшим модулем соответствующего параметра линейной модели. Обучите модели заново и оцените её качество. Замерьте скорость такого отбора признаков.



In [ ]:
#your code here

Изменилось ли качество? Как?

Подумаем, что мы не учли. Мы действовали в предположении, что признаки вносят вклад равномерно, и не учитывали их масштаб. Если мы умножим один из признаков в 100 раз, то без учёта регуляризации его вес уменьшится в эти же 100 раз. А мы на основе этого отбираем признаки! Давайте сначала отмасштабируем признаки одним из способов, а только потом будем удалять признаки.

Кстати, в таком случае надо пересчитать качество на всех признаках (сделайте это ниже). Если вы сделали нормирование признаков в самом начале, то попробуйте отобрать признаки на неотмасштабированных данных.

Что получилось?

In [ ]:
#your code here

Вопрос на засыпку: one-hot кодирование возвращает нам единичные признаки-индикаторы. Попробуйте также отскалировать их, как и обычные числовые, и снова выбрать 40 главных по вкладу признаков. Изменился ли их список? Изменится ли качество?

In [ ]:
#your code here

# Часть 3. Стекинг

Зачастую у разных моделей машинного обучения свои недостатки и преимущества при прогнозах. Возникает естественное желание объединить модели, что естественным образом приводит нас к стекингу.
<a href="https://ibb.co/96jMjYf"><img src="https://i.ibb.co/LmfGfpq/6691be7955cdd52269ee31704d7d82c0.jpg" alt="6691be7955cdd52269ee31704d7d82c0" border="0"></a>

__Важно помнить__
1. Если вы будете ансамблировать плохие модели, это ухудшит итоговое качество.
2. Важно делать предсказания out-of-fould,чтобы базовые модели не видели выборки, на которой они делают предсказания.

__Материалы__
- Два видео из реальных соревнований, в которых использовали стекинг (на самом деле, его используют везде, это лишь хорошие примеры). [Видео 1](youtube.com/watch?time_continue=225&v=np_KY9NlPuQ&embeds_referring_euri=https%3A%2F%2Fstepik.org%2F&source_ve_path=Mjg2NjY), [видео 2](https://vkvideo.ru/video-227526140_456239114?list=ln-j5YNNKEQhudtlIvNtf&t=1s&ref_domain=stepik.org).
- [Продвинутые техники стекинга](https://www.youtube.com/watch?v=7ufocNOyOQQ&embeds_referring_euri=https%3A%2F%2Fstepik.org%2F&source_ve_path=Mjg2NjY).

In [ ]:
# Стекинг через sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = ...

# Задаём базовые модели, предсказания которых будут использоваться мета-моделью
base_estimators = [
    ("Lady Gaga", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("Jony Depp", KNeighborsClassifier(n_neighbors=5)),
]

# Финальная модель
final_estimator = LogisticRegression(max_iter=1000)

# Класс стекинг
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=final_estimator,
    passthrough=False,            # если True — к признакам добавятся ещё и "сырые" X
    n_jobs=-1
)

# Обучаемся
stacking_clf.fit(X_train, y_train)

# Делаем предсказания и считаем точность
y_pred = stacking_clf.predict(X_test)
print("Точность стекинга:", accuracy_score(y_test, y_pred))


Я попросил ChatGPT реализовать простую версию стекинга. Ниже - его ответ. Найдите в нём ошибки.

In [ ]:
# стекинг: что под капотом

import numpy as np
from sklearn.base import clone

class SimpleStackingClassifier:
    def __init__(self, base_models, final_model):
        """
        base_models  — список моделей (например, [DecisionTreeClassifier(...), KNeighborsClassifier(...)]
        final_model  — одна модель, которая будет учиться на предсказаниях base_models
        """
        # Делаем копии моделей, чтобы случайно не испортить переданные снаружи объекты
        self.base_models = [clone(m) for m in base_models]
        self.final_model = clone(final_model)

    def fit(self, X, y):
        """
        Обучение стекинга:
        1) обучаем каждую базовую модель
        2) собираем их предсказания (как новые признаки)
        3) обучаем финальную модель на этих новых признаках
        """
        base_predictions = []

        # 1) обучаем базовые модели и сразу считаем их предсказания
        for model in self.base_models:
            model.fit(X, y)
            # Простая версия: берём predict (классы)
            # Можно было бы брать predict_proba, но это уже посложнее.
            preds = model.predict(X)
            base_predictions.append(preds)

        # 2) превращаем список предсказаний в матрицу признаков
        #
        meta_features = np.column_stack(base_predictions)

        # 3) обучаем финальную модель на этих "мета-признаках"
        self.final_model.fit(meta_features, y)

        return self

    def predict(self, X):
        """
        Предсказание стекинга:
        1) получаем предсказания всех базовых моделей
        2) объединяем их в признаки
        3) финальная модель выдаёт итоговый ответ
        """
        base_predictions = []

        for model in self.base_models:
            preds = model.predict(X)
            base_predictions.append(preds)

        meta_features = np.column_stack(base_predictions)

        # финальная модель выдаёт итоговое предсказание
        return self.final_model.predict(meta_features)


# Пример использования
base_models = [
    DecisionTreeClassifier(max_depth=3, random_state=42),
    KNeighborsClassifier(n_neighbors=5),
]

final_model = LogisticRegression(max_iter=1000)

my_stacking = SimpleStackingClassifier(base_models, final_model)
my_stacking.fit(X_train, y_train)

y_pred_my = my_stacking.predict(X_test)
print("Точность моего простого стекинга:", accuracy_score(y_test, y_pred_my))
